<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-08/AI_Modul_8.4_Praktikum_Interpretasi_Hasil_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 8.4: Praktikum Interpretasi Hasil Model
**Mata Kuliah:** Kecerdasan Buatan dalam Agrokompleks  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengimplementasikan algoritma **Permutation Feature Importance (PFI)** untuk meranking variabel sensor dan spektral penentu.
2. Membangun modul atribusi berbasis gradien: **Vanilla Saliency** dan **Integrated Gradients (IG)**.
3. Memverifikasi pemenuhan **Aksioma Kelengkapan (*Completeness Axiom*)** pada Integrated Gradients.
4. Mengekstrak representasi lapisan laten tersembunyi dan memproyeksikannya menggunakan **t-SNE** untuk memetakan klaster kualitas panen sawit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE

# Konfigurasi seed komputasi
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Perangkat Komputasi Aktif: {device}")


## 1. Pembangkitan Data Terpadu & Pelatihan Model JST
Kita menggunakan dataset spektral dan sensor IoT kelapa sawit (600 sampel, 124 fitur) serta melatih arsitektur JST `OilPalmYieldMLP`.


In [ ]:
n_samples = 600
n_spectral = 120
afdeling_ids = np.repeat(np.arange(1, 11), n_samples // 10)

wavelengths = np.linspace(400, 1000, n_spectral)
X_spectral = np.zeros((n_samples, n_spectral))
for i in range(n_samples):
    base_sig = 0.15 + 0.35 * np.exp(-((wavelengths - 550)**2) / (2 * 45**2)) +                       0.50 * (1.0 / (1.0 + np.exp(-(wavelengths - 700) / 20)))
    noise = np.random.normal(0, 0.02, n_spectral)
    X_spectral[i] = base_sig * np.random.uniform(0.85, 1.15) + noise

pH = np.random.normal(5.2, 0.4, (n_samples, 1))
nitrogen = np.random.normal(120.0, 25.0, (n_samples, 1))
moisture = np.random.uniform(30.0, 70.0, (n_samples, 1))
rainfall = np.random.uniform(100.0, 350.0, (n_samples, 1))
X_sensor = np.hstack([pH, nitrogen, moisture, rainfall])
X_raw = np.hstack([X_spectral, X_sensor])

y_raw = 19.5 + 4.5 * X_spectral[:, 65] - 0.015 * np.abs(pH[:, 0] - 5.5)**2 +         0.012 * (nitrogen[:, 0] / 10.0) + np.random.normal(0, 0.35, n_samples)
y_raw = y_raw.reshape(-1, 1)

train_mask = np.isin(afdeling_ids, [1, 2, 3, 4, 5, 6])
test_mask  = np.isin(afdeling_ids, [9, 10])

mu_tr = np.mean(X_raw[train_mask], axis=0, keepdims=True)
std_tr = np.std(X_raw[train_mask], axis=0, keepdims=True) + 1e-8

X_train_norm = (X_raw[train_mask] - mu_tr) / std_tr
X_test_norm  = (X_raw[test_mask] - mu_tr) / std_tr

y_train = y_raw[train_mask]
y_test  = y_raw[test_mask]

class AgroDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(AgroDataset(X_train_norm, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(AgroDataset(X_test_norm, y_test), batch_size=32, shuffle=False)

# Arsitektur JST Terstruktur
class OilPalmYieldMLP(nn.Module):
    def __init__(self, in_features=124, hidden1=64, hidden2=32, out_features=1):
        super(OilPalmYieldMLP, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(p=0.15),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(p=0.10)
        )
        self.regressor = nn.Linear(hidden2, out_features)
        
    def forward(self, x):
        h = self.feature_extractor(x)
        return self.regressor(h)

model = OilPalmYieldMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.MSELoss()

# Pelatihan 30 Epoch
model.train()
for epoch in range(30):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

print("Model JST siap dibongkar menggunakan modul XAI!")


## 2. Permutation Feature Importance (PFI)
Kita mengacak satu demi satu kolom fitur masukan pada data uji, kemudian mengukur kenaikan galat RMSE ($\Delta \text{RMSE}$).


In [ ]:
model.eval()
X_test_tensor = torch.tensor(X_test_norm, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

with torch.no_grad():
    base_preds = model(X_test_tensor)
    base_rmse = torch.sqrt(criterion(base_preds, y_test_tensor)).item()

n_feats = X_test_tensor.shape[1]
pfi_scores = np.zeros(n_feats)

for j in range(n_feats):
    X_perm = X_test_tensor.clone()
    perm_idx = torch.randperm(X_perm.size(0))
    X_perm[:, j] = X_perm[perm_idx, j]
    
    with torch.no_grad():
        perm_preds = model(X_perm)
        perm_rmse = torch.sqrt(criterion(perm_preds, y_test_tensor)).item()
        
    pfi_scores[j] = perm_rmse - base_rmse

# Label Nama Fitur
feature_names = [f"Spektral {int(w)}nm" for w in wavelengths] + ['pH Tanah', 'Nitrogen Tanah', 'Kelembaban Tanah', 'Curah Hujan']

# Ambil 10 Fitur Terpenting
top_indices = np.argsort(pfi_scores)[::-1][:10]

print("=================================================================")
print("        TOP 10 FITUR PALING BERPENGARUH (PFI RANKING)            ")
print("=================================================================")
for rank, idx in enumerate(top_indices, 1):
    print(f"Peringkat {rank:02d}: {feature_names[idx]:<25} | Kenaikan RMSE: +{pfi_scores[idx]:.4f}")
print("=================================================================")


## 3. Atribusi Fitur Berbasis Gradien: Integrated Gradients
Kita mengintegrasikan vektor gradien di sepanjang lintasan linier antara titik acuan rata-rata (baseline) dan sampel daun sawit aktual.


In [ ]:
def integrated_gradients(model, x_input, x_baseline, m_steps=40):
    accum_grads = torch.zeros_like(x_input)
    for k in range(1, m_steps + 1):
        alpha = float(k) / m_steps
        x_step = x_baseline + alpha * (x_input - x_baseline)
        x_step.requires_grad_(True)
        
        out = model(x_step)
        grad = torch.autograd.grad(out, x_step)[0]
        accum_grads += grad
        
    avg_grads = accum_grads / m_steps
    attributions = (x_input - x_baseline) * avg_grads
    return attributions

# Pilih sampel pohon ke-0 dari data uji
sample_x = X_test_tensor[0:1]
baseline_x = torch.zeros_like(sample_x) # Baseline rata-rata (0.0 pada Z-Score)

ig_attr = integrated_gradients(model, sample_x, baseline_x, m_steps=50)
ig_attr_np = ig_attr.detach().cpu().numpy().flatten()

# Verifikasi Aksioma Kelengkapan (Completeness Axiom)
with torch.no_grad():
    pred_sample = model(sample_x).item()
    pred_baseline = model(baseline_x).item()
    delta_pred = pred_sample - pred_baseline

sum_attributions = np.sum(ig_attr_np)
gap_error = abs(delta_pred - sum_attributions)

print("=================================================================")
print("           VERIFIKASI AKSIOMA KELENGKAPAN (COMPLETENESS)         ")
print("=================================================================")
print(f"Prediksi Sampel Aktual f(x)       : {pred_sample:.4f} % Rendemen")
print(f"Prediksi Titik Baseline f(x')     : {pred_baseline:.4f} % Rendemen")
print(f"Selisih Prediksi [f(x) - f(x')]   : {delta_pred:.4f}")
print(f"Total Jumlahan Atribusi (Sum IG)  : {sum_attributions:.4f}")
print(f"Selisih Galat Kelengkapan (Gap)   : {gap_error:.6f}")
print("=================================================================")
assert gap_error < 0.05, "Aksioma kelengkapan terpenuhi secara presisi!"
print("[OK] Aksioma Kelengkapan Terpenuhi!")


## 4. Visualisasi Profil Atribusi Spektral dan Ranking Fitur
Kita menampilkan profil kontur atribusi panjang gelombang spektrometri serta ranking 10 fitur penentu.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: PFI Bar Chart
top_pfi = pfi_scores[top_indices]
top_names = [feature_names[i] for i in top_indices]
y_positions = np.arange(len(top_names))

ax1.barh(y_positions, top_pfi, color='teal', edgecolor='black', alpha=0.8)
ax1.set_yticks(y_positions)
ax1.set_yticklabels(top_names, fontsize=9)
ax1.invert_yaxis()
ax1.set_title('A. Ranking 10 Fitur Paling Signifikan (PFI)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Kenaikan Galat RMSE Pasca-Pengacakan Fitur')
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot 2: Profil Atribusi Spektral (Integrated Gradients)
spectral_attr = ig_attr_np[:n_spectral]
ax2.plot(wavelengths, spectral_attr, 'b-', lw=1.8, label='Skor Atribusi IG')
ax2.axvspan(680, 750, color='red', alpha=0.18, label='Pita Red-Edge Klorofil (700-740 nm)')
ax2.axvspan(920, 980, color='blue', alpha=0.12, label='Pita Serapan Kadar Air (Water Band)')
ax2.set_title('B. Profil Atribusi Spektral Vis-NIR pada Satu Sampel Pohon', fontsize=11, fontweight='bold')
ax2.set_xlabel('Panjang Gelombang (nm)')
ax2.set_ylabel('Skor Atribusi Integrated Gradients')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 5. Eksplorasi Ruang Representasi Laten (t-SNE)
Kita mengekstrak vektor aktivasi lapisan tersembunyi terakhir (32 dimensi) dari seluruh data uji, lalu memproyeksikannya ke ruang 2 dimensi menggunakan t-SNE.


In [ ]:
# Ekstraksi aktivasi lapisan tersembunyi
feature_extractor = model.feature_extractor
feature_extractor.eval()

with torch.no_grad():
    latent_reps = feature_extractor(X_test_tensor).cpu().numpy()

# Proyeksi t-SNE
tsne = TSNE(n_components=2, perplexity=20, random_state=42)
latent_2d = tsne.fit_transform(latent_reps)

# Kategorisasi Rendemen untuk Pewarnaan
cpo_actual = y_test.flatten()
categories = []
for val in cpo_actual:
    if val < 21.5:
        categories.append('Rendah (<21.5%)')
    elif val <= 23.0:
        categories.append('Sedang (21.5-23.0%)')
    else:
        categories.append('Tinggi (>23.0%)')

categories = np.array(categories)

plt.figure(figsize=(8, 6))
colors = {'Rendah (<21.5%)': '#E53E3E', 'Sedang (21.5-23.0%)': '#DD6B20', 'Tinggi (>23.0%)': '#38A169'}

for cat, col in colors.items():
    mask = categories == cat
    plt.scatter(latent_2d[mask, 0], latent_2d[mask, 1], label=cat, color=col, alpha=0.85, s=60, edgecolors='k')

plt.title('Proyeksi Ruang Laten JST (t-SNE) Menunjukkan Klaster Alami Rendemen', fontsize=11, fontweight='bold')
plt.xlabel('Dimensi Laten t-SNE 1')
plt.ylabel('Dimensi Laten t-SNE 2')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print("[OK] Sesi Praktikum Explainable AI (XAI) Selesai!")
